In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ast

In [ ]:
movies_df = pd.read_csv('../data/movie.metadata.tsv', sep='\t', header=None)
characters_df = pd.read_csv('../data/character.metadata.tsv', sep='\t', header=None)
plot_summaries_df = pd.read_csv('../data/plot_summaries.txt', sep='\t', header=None)
names_df = pd.read_csv('../data/name.clusters.txt', sep='\t', header=None)



movies_df.columns = ['wikipedia_id', 'freebase_id', 'name', 'release_date', 'revenue', 'runtime', 'languages', 'countries', 'genres']

characters_df.columns = ['wikipedia_id', 'freebase_id', 'release_date', 'character_name', 'actor_date_of_birth', 
                         'actor_gender', 'actor_height', 'actor_ethnicity', 'actor_name', 'actor_age_at_movie_release',
                         'freebase_character_actor_map_id', 'freebase_character_id', 'freebase_actor_id']

plot_summaries_df.columns = ['wikipedia_id', 'plot_summary']

names_df.columns = ['character_name', 'freebase_id']

In [ ]:
def extract_data(genre_string):
    # Safely evaluate the string to a dictionary
    genre_dict = ast.literal_eval(genre_string)
    # Extract only the values (genre names) and return as a list
    return list(genre_dict.values())

movies_df = movies_df.dropna(subset=['release_date'])
movies_df['genres'] = movies_df['genres'].apply(extract_data)
movies_df['languages'] = movies_df['languages'].apply(extract_data)
movies_df['countries'] = movies_df['countries'].apply(extract_data)
# keep only the year of the release date (4 first characters)
movies_df['release_date'] = movies_df['release_date'].astype(str).apply(lambda x: x[:4]).astype(int)
movies_df.head()

In [ ]:
characters_df.head()

In [ ]:
plot_summaries_df.head()

In [ ]:
movies_with_plot_df = pd.merge(movies_df, plot_summaries_df, on='wikipedia_id', how='inner')
movies_with_plot_df.head()

In [ ]:
# Cleaning
clean_df = movies_with_plot_df.dropna(subset=['name', 'wikipedia_id', 'release_date', 'genres', 'countries', 'languages', 'plot_summary'])
clean_df = clean_df.reset_index(drop=True)
clean_df = clean_df[clean_df['release_date'] > 1900]

usa_df = clean_df[clean_df['countries'].apply(lambda x: 'United States' in x)]

clean_after_45_df = clean_df[clean_df['release_date'] > 1945]
number_characters_movie_plot = clean_after_45_df['plot_summary'].apply(lambda x: len(x.split()))
number_characters_movie_plot.sum()

In [ ]:
plt.figure(figsize=(12,6))
plt.hist(clean_df['release_date'], bins=100, edgecolor='black', alpha=0.7)
plt.title(f'Distribution of USA films')
plt.xlabel('Release Year')
plt.ylabel('Frequency')
# draw red on the cold war period
plt.axvline(x=1947, color='red', linestyle='--')
plt.axvline(x=1991, color='red', linestyle='--')
plt.grid(True)
plt.show()

In [ ]:
# movie_name = "First Blood"
# movie_name = "The Good, the Bad and the Ugly"
movie_name = "High School Musical"
# movie_name = "From Russia with Love"
# movie_name = "Oshibka rezidenta"
movie = clean_df[clean_df['name'] == movie_name]
print(movie['plot_summary'].iloc[0])
print(movie['release_date'].iloc[0])


# print movies name that are not unique
movies_name_counts = clean_df['wikipedia_id'].value_counts()
movies_name_counts[movies_name_counts > 1]
# Prompt:
# Here is a movie plot. You first need to identify if it can be identified to the Eastern or Western bloc during the Cold War. If yes come up with the character or group of character impersonating the Western and Eastern bloc and their values.
# Your output needs to be parsable with the first line being the first character followed by it's characteristics each comma separated.
# Second line on the second character.
# Then the theme of the movies and keywords comma separated.
# Then Cold war side belonging either Easter, Western or None.
# Then Western bloc represensation comma separated main values and characteristics 
# Then Eastern bloc representation


# Result:


In [ ]:
import os
from openai import AzureOpenAI
import keyring


endpoint = keyring.get_password("AzureOpenAI", "endpoint")
key = keyring.get_password("AzureOpenAI", "key")

model_name = "gpt-4o"

client = AzureOpenAI(azure_endpoint=endpoint,api_version="2024-08-01-preview",api_key=key)

In [ ]:
prompt = "Here is a movie plot. You first need to identify if it can be identified to the Eastern or Western bloc during the Cold War. If yes come up with the character or group of character impersonating the Western and Eastern bloc and their values. \n \
Your output needs to be parsable with the first line being the first character followed by it's characteristics each comma separated. \n \
Second line on the second character. \n \
Then the theme of the movies and keywords comma separated. \n \
Then Cold war side belonging either Easter, Western or None. \n \
Then Western bloc represensation comma separated main values and characteristics \n \
Then Eastern bloc representation \n" + "year: " + str(movie['release_date'].iloc[0]) + "\nplot: " + movie['plot_summary'].iloc[0]

# You are an expert in movie history and Cold War. You will be given the name of the film, the year and the plot of the movie. You first need to analyse if the movie can be identified to the Eastern or Western bloc during the Cold War. If yes come up with the character or group of character impersonating the Western and Eastern bloc and their values as well as their main archetye.

# Your output needs to be parsable with the following form without context just line after line only the keyword corresponding:
# Cold War side belonging either Easter, Western or None.
# The character or group of character representing Western bloc with their values and archetype comma separated.
# The character or group of character representing Eastern bloc with their values and archetype comma separated.
# The Western bloc representation comma separated main values and characteristics 
# The Eatern bloc representation comma separated main values and characteristics 
# The theme of the movies and keywords comma separated.


prompt = "You are an expert in movie history and Cold War. You will be given the name of the film, the year and the plot of the movie. You first need to analyse if the movie can be identified to the Eastern or Western bloc during the Cold War. If yes come up with the character or group of character impersonating the Western and Eastern bloc and their values as well as their main archetye.\n\
Your output needs to be parsable with the following form without explanation of what it corresponds to just line after line only the keyword corresponding to:\n\
Cold War side belonging either Easter, Western or None.\n\
The character or group of character representing Western bloc with their values and archetype comma separated.\n\
The character or group of character representing Eastern bloc with their values and archetype comma separated.\n\
The Western bloc representation comma separated main values and characteristics\n\
The Eatern bloc representation comma separated main values and characteristics\n\
The theme of the movies and keywords comma separated."

def create_query_string(prompt, movie_name, movie_year, movie_plot):
    return prompt + "movie: " + movie_name + "\nyear: " + str(movie_year) + "\nplot: " + movie_plot

response = client.chat.completions.create(
    model=model_name, # replace with the model deployment name of your o1-preview, or o1-mini model
    messages=[
        {"role": "user", "content": prompt},
    ],
)

print(response.model_dump_json(indent=2))

In [ ]:
print(response.choices[0].message.content)

In [ ]:
import spacy
from textblob import TextBlob

# Load SpaCy model for NER and dependency parsing
nlp = spacy.load("en_core_web_sm")

# Movie plot summary to analyze
first_summary = "During World War II, the Allies face the Axis powers in a brutal conflict. The Allies, depicted as courageous and resilient, fight against tyranny, while the Axis is portrayed as ruthless and oppressive."

# Custom list of known belligerents for historical contexts
known_belligerents = ["Allies", "Axis"]

# Dictionary to store portrayals
belligerents = {key: {"descriptions": [], "sentiment": 0} for key in known_belligerents}

# Process the text
doc = nlp(first_summary)

# Analyze sentences for descriptions and sentiment
for sent in doc.sents:
    for belligerent in known_belligerents:
        if belligerent in sent.text:
            # Sentiment analysis
            blob = TextBlob(sent.text)
            belligerents[belligerent]["sentiment"] += blob.sentiment.polarity
            
            # Extract descriptive words related to the belligerent
            for token in sent:
                if token.dep_ in ["amod", "acomp", "attr"]:
                    belligerents[belligerent]["descriptions"].append(token.text)

# Display results
for belligerent, info in belligerents.items():
    descriptions = ", ".join(info["descriptions"])
    sentiment = "Positive" if info["sentiment"] > 0 else "Negative" if info["sentiment"] < 0 else "Neutral"
    print(f"{belligerent} portrayal:\n- Descriptions: {descriptions}\n- Sentiment: {sentiment}")